# Retail retention and repeat-purchase targeting

**Daniel Umeadi — SQL + Python**

Which existing customers should be prioritized for a measured repeat-purchase campaign?

Full UCI Online Retail, Chen (2015), CC BY 4.0. Source dates: 2010-12-01 to 2011-12-09. Analytical window ends before 2011-12-01.

## 1. Rebuild the analysis

Run the audited pipeline from this notebook directory. Source data is downloaded only when absent; hashes are checked before use.

In [1]:
import sys, subprocess
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
completed = subprocess.run([sys.executable, 'run.py'], check=True, capture_output=True, text=True)
print(completed.stdout)


{
  "positive_value_gbp": 10027891.863998253,
  "customers": 4297,
  "weighted_month_one_retention": 0.2365374937091092,
  "test_month": "2011-11",
  "train_snapshot_months": [
    "2011-08",
    "2011-09",
    "2011-10"
  ],
  "test_customers": 3974,
  "purchase_prevalence": 0.33744338198288876,
  "average_precision": 0.634419719775543,
  "roc_auc": 0.7346935712135283,
  "brier": 0.18630388323206729,
  "constant_train_rate_brier": 0.22789763864528512,
  "top_20pct_precision": 0.6729559748427673,
  "top_20pct_lift": 1.9942781834639502,
  "recency_rule_top20_lift": 1.4276795219938188,
  "top_group_n": 795
}



## 2. Inspect the result and audit

Read denominators and exclusions before interpreting the findings.

In [2]:
summary=json.loads(Path('outputs/summary.json').read_text())
display(pd.Series(summary,name='value').to_frame())
checks=json.loads(Path('outputs/quality_checks.json').read_text())
display(pd.Series(checks,name='value').to_frame())

,value
positive_value_gbp,10027891.863998
customers,4297
weighted_month_one_retention,0.236537
test_month,2011-11
train_snapshot_months,"[2011-08, 2011-09, 2011-10]"
test_customers,3974
purchase_prevalence,0.337443
average_precision,0.63442
roc_auc,0.734694
brier,0.186304


,value
source_rows,541909
window_rows,516384
excluded_partial_december_rows,25525
missing_customer_rows_source,135080
exact_duplicate_rows_retained_source,5268
source_date_min,2010-12-01 08:26:00
source_date_max,2011-12-09 12:50:00
positive_purchase_rows,504993
positive_value_gbp,10027891.863998
identified_positive_value_share,0.836987


## 3. Inspect the analytical output

In [4]:
display(pd.read_csv('outputs/segments.csv'))


,segment,customers,positive_value_gbp,median_recency_days
0,Recent repeat,1143,5663850.971,11.0
1,Other repeat,1336,1648300.202,55.0
2,One observed order,1538,631751.511,125.0
3,Lapsed repeat,280,449312.430,141.0


## 4. Interpretation

4,297 identified customers; 23.65% weighted month-one retention. November top-fifth propensity lift is 1.99×, against 1.43× for a simple recency rule. Average precision: 0.634; holdout purchase prevalence: 0.337.

**Limits:** No treatment effect is measured. Customer IDs can be missing. Acquisition dates precede observation for some customers. Identical lines are retained; invoice value is not profit. Model calibration is imperfect in the November holiday period.

**Next decision:** Which existing customers should be prioritized for a measured repeat-purchase campaign? 

## Charts

![01_sales](outputs/01_sales.png)

![02_cohorts](outputs/02_cohorts.png)

![03_segments](outputs/03_segments.png)

![04_holdout](outputs/04_holdout.png)

